# Notebook 07 - Sensitivity Analysis (Per-Layer + Per-Expert)

## Learning objective
Identify which parts of DeepSeek-Coder-V2-Lite-Instruct are most sensitive to quantization, so we can make principles mixed-precision decisions in Notebook 08.

## Key concepts
- **Sensitivity**: how much quality drops when a specific component is quantized while others stay higher precision.
- **Per-layer sensitivity**: some transformer blocks tolerate INT4 well, others cause large degredations
- **Per-expert sensitivity**: experts are not equally important; some may be much more fragile under quantization
- **Ablation thinking**: change one component at a time, measure effect, rank impact.
- **Heatmap interpretation**: darker/higher values = larger performance drop = more precision-sensitive component.

## Impact
This is the core intellectual step of the project: moving from "INT4 vs INT8 comparison" to "which internal components actually drive that gap."

## Inputs from previous notebooks
- `results/03_baseline_evaluation.json` (FP16 quality reference)
- `results/05_ptq_artifacts_summary.json` (quantized artifact metadata)
- `results/06_benchmarksing_perf_snapshot.json` (cross-precision speed/memory context)

## Outputs we will produce
- Layer sensitivity table + ranked list
- Expert sensitivity table + ranked list
- Layer/expert heatmaps saved to `results/`
- A recommended precision policy for Notebook 08

# TODO
- [ ] Clean up imports.  No duplicates
- [ ] Refactor repeated code chunks to functions, where appropriate
- [ ] Add explicit memory management after compute-heavy cells

In [1]:
# Imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
from llama_cpp import Llama                                                                                              
from datasets import load_dataset
import pandas as pd
import datetime as dt
import numpy as np
import json
import gc
import time
import re

In [ ]:
# Setup: load benchmark context from Notebook 06
ROOT = Path.cwd().resolve().parent
RESULTS_DIR = ROOT / "results"
BENCHMARK_PATH = RESULTS_DIR / "06_benchmarking_perf_snapshot.json"

with BENCHMARK_PATH.open() as f:
    bench = json.load(f)

metadata = bench["benchmark_results"]["metadata"]
artifacts = bench["benchmark_results"]["artifacts"]
measurements = bench["benchmark_results"]["measurements"]

summary_rows = []

for precision, artifact_info in artifacts.items():
    perf = measurements.get(precision, {}).get("performance", {})
    mem = measurements.get(precision, {}).get("memory", {})
    summary_rows.append(
        {
            "precision": precision,
            "artifact_size_gib": artifact_info.get("artifact_size_gib"),
            "tokens_per_sec_mean": perf.get("tokens_per_sec_mean"),
            "ttft_sec_mean": perf.get("ttft_sec_mean"),
            "peak_rs_gib": mem.get("peak_rss_gib"),
        }
    )

benchmark_summary_df = pd.DataFrame(summary_rows).sort_values("artifact_size_gib", ascending=False)

print(f"Loaded benchmark run: {metadata.get('run_id')}")
print(f"Model: {metadata.get('model_id')}")
display(benchmark_summary_df)

Loaded benchmark run: 202604d06T025654Z
Model: deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct


,precision,artifact_size_gib,tokens_per_sec_mean,ttft_sec_mean,peak_rs_gib
0,fp16,29.27,74.738630,0.056434,29.451981
1,q8_0,15.56,114.013102,0.050479,38.823807
2,q4_k_m,9.66,137.556743,0.041548,9.443710


Load baseline references and define one shared sensitivity metric for this notebook.
Higher delta means more quantization damage relative to fp16 baseline

In [3]:
BASELINE_PATH = RESULTS_DIR / "03_baseline_evaluation.json"
EXPLORATION_PATH = RESULTS_DIR / "01_model_exploration.json"

with BASELINE_PATH.open() as f:
    baseline = json.load(f)
with EXPLORATION_PATH.open() as f:
    arch = json.load(f)

baseline_pass1 = float(baseline["summary"]["pass_at_1_percent"])
n_humaneval = int(baseline["summary"]["n_total"])
fp16_tps = float(bench["benchmark_results"]["measurements"]["fp16"]["performance"]["tokens_per_sec_mean"])

def sensitivity_delta_pass1(candidate_pass1_percent: float, reference_pass1_percent: float = baseline_pass1) -> float:
    return float(reference_pass1_percent - candidate_pass1_percent)

def sensitivity_rel_drop(candidate_pass1_percent: float, reference_pass1_percent: float = baseline_pass1) -> float:
    if reference_pass1_percent == 0:
        return np.nan
    return float((reference_pass1_percent - candidate_pass1_percent) / reference_pass1_percent)

context_df = pd.DataFrame(
    [
        {"metric": "baseline_pass@1_percent", "value": baseline_pass1},
        {"metric": "humaneval_n_problems", "value": n_humaneval},
        {"metric": "fp16_tokens_per_sec_mean", "value": fp16_tps},
        {"metric": "num_layers", "value": int(arch["num_layers"])},
        {"metric": "num_moe_layers", "value": len(arch["moe_layers"])},
        {"metric": "n_routed_experts", "value": int(arch["n_routed_experts"])},
        {"metric": "n_shared_experts", "value": int(arch["n_shared_experts"])},
    ]
)

display(context_df)

,metric,value
0,baseline_pass@1_percent,60.365854
1,humaneval_n_problems,164.000000
2,fp16_tokens_per_sec_mean,74.738630
3,num_layers,27.000000
4,num_moe_layers,26.000000
5,n_routed_experts,64.000000
6,n_shared_experts,2.000000


Build a first-pass ablation manifest (coarse layer sweeep + expert sample sweep) for Notebook 07
We start coarse to map the landscape, then refine around the most sensitive regions

In [6]:
num_layers = int(arch["num_layers"])
moe_layers = list(map(int, arch["moe_layers"]))
n_routed_experts = int(arch["n_routed_experts"])

n_layer_groups = 9
layer_groups = np.array_split(np.arange(num_layers), n_layer_groups)

layer_manifest = []
for i, group in enumerate(layer_groups):
    layer_manifest.append(
        {
            "ablation_id": f"layer_group_{i:02d}",
            "kind": "layer_group",
            "target_layers": [int(x) for x in group.tolist()],
            "target_experts": None,
            "quantize_to": "q4_k_m",
            "keep_rest_as": "fp16",
            "status": "pending",
            "candidate_pass_at_1_percent": None,
            "delta_pass_at_1_percent": None,
            "relative_drop": None,
            "notes": "",
        }
    )

# first pass over every 4th routed expert across all MoE layers
expert_stride = 4
expert_sample_ids = list(range(0, n_routed_experts, expert_stride))

expert_manifest = []
for expert_id in expert_sample_ids:
    expert_manifest.append(
        {
            "ablation_id": f"expert_{expert_id:02d}_across_all_moe_layers",
            "kind": "expert_across_layers",
            "target_layers": moe_layers,
            "target_experts": [int(expert_id)],
            "quantize_to": "q4_k_m",
            "keep_rest_as": "fp16",
            "status": "pending",
            "candidate_pass_at_1_percent": None,
            "delta_pass_at_1_percent": None,
            "relative_drop": None,
            "notes": "",
        }
    )

manifest = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "model_id": baseline["metadata"]["model_id"],
    "reference": {
        "baseline_pass_at_1_percent": baseline_pass1,
        "baseline_source": str(BASELINE_PATH),
    },
    "design": {
        "phase_1_layer_groups": n_layer_groups,
        "phase_1_expert_stride": expert_stride,
        "phase_2_rule": "Refine around top-3 most sensitive layer groups and top-8 most sensitive experts from phase 1.",
    },
    "ablations": {
        "layer": layer_manifest,
        "expert": expert_manifest,
    },
}

MANIFEST_PATH = RESULTS_DIR / "07_sensitivity_manifest.json"
with MANIFEST_PATH.open("w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote manifest scaffold: {MANIFEST_PATH}")
print(f"Layer ablations: {len(layer_manifest)} | Expert ablations: {len(expert_manifest)}")

preview_df = pd.DataFrame(layer_manifest[:3] + expert_manifest[:3])[
    ["ablation_id", "kind", "quantize_to", "status"]
]
display(preview_df)

Wrote manifest scaffold: /Users/jarrett/dev/quantization-study/results/07_sensitivity_manifest.json
Layer ablations: 9 | Expert ablations: 16


,ablation_id,kind,quantize_to,status
0,layer_group_00,layer_group,q4_k_m,pending
1,layer_group_01,layer_group,q4_k_m,pending
2,layer_group_02,layer_group,q4_k_m,pending
3,expert_00_across_all_moe_layers,expert_across_layers,q4_k_m,pending
4,expert_04_across_all_moe_layers,expert_across_layers,q4_k_m,pending
5,expert_08_across_all_moe_layers,expert_across_layers,q4_k_m,pending


Claim the next pending ablation and define a helper to record its measured pass@1.

This keeps the manifest as the single source of truth for run status and sensitivity deltas.

In [7]:
with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

baseline_pass1 = float(manifest["reference"]["baseline_pass_at_1_percent"])

all_items = manifest["ablations"]["layer"] + manifest["ablations"]["expert"]
pending = [a for a in all_items if a["status"] == "pending"]

if not pending:
    print("No pending ablations left.")
    current = None
else:
    current = pending[0]
    current["status"] = "in_progress"
    current["started_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
    with MANIFEST_PATH.open("w") as f:
        json.dump(manifest, f, indent=2)

    print("Claimed ablation:")
    print(f"   id:            {current['ablation_id']}")
    print(f"   kind:          {current['kind']}")
    print(f"   target_layers: {current['target_layers']}")
    print(f"   target_experts:{current['target_experts']}")
    print(f"   quantize_to:   {current['quantize_to']}")
    print("")
    print("After you run evaluation for this ablation, call:")
    print("record_ablation_result(candidate_pass1_percent=<float>, notes='<optional>')")

def record_ablation_result(candidate_pass1_percent: float, notes: str = ""):
    with MANIFEST_PATH.open() as f:
        m = json.load(f)

    target = None
    for section in ["layer", "expert"]:
        for a in m["ablations"][section]:
            if a.get("status") == "in_progress":
                target = a
                break
        if target is not None:
            break
    
    if target is None:
        raise RuntimeError("No ablation is currently in_progress.")
    
    candidate = float(candidate_pass1_percent)
    delta = baseline_pass1 - candidate
    rel_drop = (delta / baseline_pass1) if baseline_pass1 else None

    target["candidate_pass_at_1_percent"] = candidate
    target["delta_pass_at_1_percent"] = delta
    target["relative_drop"] = rel_drop
    target["status"] = "completed"
    target["completed_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
    if notes:
        target["notes"] = notes

    with MANIFEST_PATH.open("w") as f:
        json.dump(m, f, indent=2)

    print("Recorded result:")
    print(f"  id: {target['ablation_id']}")
    print(f"  candidate_pass@1: {candidate:.4f}")
    print(f"  delta_pass@1:     {delta:.4f}")
    print(f"  relative_drop:    {rel_drop:.4%}" if rel_drop is not None else "  relative_drop:    None")

Claimed ablation:
   id:            layer_group_00
   kind:          layer_group
   target_layers: [0, 1, 2]
   target_experts:None
   quantize_to:   q4_k_m

After you run evaluation for this ablation, call:
record_ablation_result(candidate_pass1_percent=<float>, notes='<optional>')


Heavy cell: load model/tokenizer once for Notebook 7 eval.

In [8]:
BASELINE_PATH = ROOT / "results" / "03_baseline_evaluation.json"

with BASELINE_PATH.open() as f:
    baseline = json.load(f)

model_id = baseline["metadata"]["model_id"]

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

if "model" not in globals():
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

print(f"Loaded tokenizer/model for: {model_id}")

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1
`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

Loaded tokenizer/model for: deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct


Run HumanEval for the currently in-progress ablation and compute candidate_pass1_percent.

Requres notebook-03 evaluation functions to already exist in this kernel (evaluate_problem + dependencies)

In [9]:
# Need to reset path to import helper functions
import sys                                                                                                               
from pathlib import Path                                                                                                 
                                                                                                                        
PROJECT_ROOT = Path.cwd().resolve().parent  # notebook is in notebooks/, root is one level up                            
if str(PROJECT_ROOT) not in sys.path:                                                                                    
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.humaneval_helpers import evaluate_problem, clean_and_extract, execute_with_timeout

# Reset original root
ROOT = Path.cwd().resolve().parent

In [10]:
with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

active = None
for section in ["layer", "expert"]:
    for a in manifest["ablations"][section]:
        if a.get("status") == "in_progress":
            active = a
            break
    if active is not None:
        break

if active is None:
    raise RuntimeError("No ablation is in_progress. Claim one first.")

humaneval = load_dataset("openai/openai_humaneval", split="test")
max_problems = 164 # set smaller (e.g. 20) for quick smoke test
subset = humaneval.select(range(min(max_problems, len(humaneval))))

ablation_results = []
t0 = time.time()

for i, problem in enumerate(subset, start=1):
    p0 = time.time()
    r = evaluate_problem(problem, model=model, tokenizer=tokenizer)
    elapsed = time.time() - p0
    r["elapsed_sec"] = elapsed
    ablation_results.append(r)

    if i % 10 == 0 or i == len(subset):
        n_pass = sum(1 for x in ablation_results if x["result"] == "passed")
        print(f"[{i:3d}/{len(subset)}] running pass rate: {100*n_pass/i:.2f}%")

total_elapsed = time.time() - t0
n_total = len(ablation_results)
n_passed = sum(1 for r in ablation_results if r["result"] == "passed")
candidate_pass1_percent = 100.0 * n_passed / n_total

out = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "ablation_id": active["ablation_id"],
    "n_total": n_total,
    "n_passed": n_passed,
    "candidate_pass_at_1_percent": candidate_pass1_percent,
    "runtime_sec_total": total_elapsed,
    "results": ablation_results,
}
out_path = RESULTS_DIR / f"07_{active['ablation_id']}_humaneval.json"
with out_path.open("w") as f:
    json.dump(out, f, indent=2)

print("\nDone.")
print(f"ablation_id: {active['ablation_id']}")
print(f"candidate_pass1_percent: {candidate_pass1_percent:.6f}")
print(f"saved: {out_path}")
print("\nNow run:")
print(f"record_ablation_result(candidate_pass1_percent={candidate_pass1_percent:.6f}, notes='full HumanEval run')")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[ 10/164] running pass rate: 50.00%
[ 20/164] running pass rate: 65.00%
[ 30/164] running pass rate: 70.00%
[ 40/164] running pass rate: 67.50%
[ 50/164] running pass rate: 70.00%
[ 60/164] running pass rate: 65.00%
[ 70/164] running pass rate: 62.86%
[ 80/164] running pass rate: 63.75%
[ 90/164] running pass rate: 63.33%
[100/164] running pass rate: 62.00%
[110/164] running pass rate: 60.91%
[120/164] running pass rate: 57.50%
[130/164] running pass rate: 53.85%
[140/164] running pass rate: 50.00%
[150/164] running pass rate: 50.00%
[160/164] running pass rate: 50.00%
[164/164] running pass rate: 49.39%

Done.
ablation_id: layer_group_00
candidate_pass1_percent: 49.390244
saved: /Users/jarrett/dev/quantization-study/results/07_layer_group_00_humaneval.json

Now run:
record_ablation_result(candidate_pass1_percent=49.390244, notes='full HumanEval run')


In [11]:
record_ablation_result(candidate_pass1_percent=49.390244, notes='full HumanEval run')

Recorded result:
  id: layer_group_00
  candidate_pass@1: 49.3902
  delta_pass@1:     10.9756
  relative_drop:    18.1818%


#### Setting up the pipeline
Evaluate any GGUF artifact (fp16/q8_0/q4_k_m) with the same llama_cpp backend and same harnes.

This gives backend parity so precision is the main variable changing across runs.

In [12]:
with BENCHMARK_PATH.open() as f:
    bench = json.load(f)

def run_humaneval_gguf(model_key="fp16", max_problems=164, timeout_sec=10):
    model_path = bench["benchmark_results"]["artifacts"][model_key]["artifact_path"]
    llm = Llama(model_path=model_path, n_ctx=2048, n_threads=8, n_gpu_layers=-1, verbose=False)

    def generate_completion_gguf(problem):
        prompt = (
            "Complete the following Python function. Return raw Python code only with correct "
            "newlines and indentation. Do not use markdown fences or explanations.\n\n"
            + problem["prompt"]
        )
        out = llm(prompt, max_tokens=512, temperature=0.0, top_p=1.0)
        text = out["choices"][0]["text"]
        return text.replace("\r\n", "\n").replace("\r", "\n").replace("Ġ", " ").replace("Ċ", "\n").rstrip()
    
    ds = load_dataset("openai/openai_humaneval", split="test")
    subset = ds.select(range(min(max_problems, len(ds))))

    results, t0 = [], time.time()
    for i, problem in enumerate(subset, start=1):
        p0 = time.time()
        generated = generate_completion_gguf(problem)
        code = clean_and_extract(generated, problem)
        full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})"
        result = execute_with_timeout(full_code, timeout=timeout_sec)
        results.append({
            "task_id": problem["task_id"],
            "result": result,
            "generated": generated,
            "elapsed_sec": time.time() - p0,
        })
        if i % 10 == 0 or i == len(subset):
            n_pass = sum(1 for r in results if r["result"] == "passed")
            print(f"[{i:3d}/{len(subset)}] running pass rate: {100*n_pass/i:.2f}%")

    n_total = len(results)
    n_passed = sum(1 for r in results if r["result"] == "passed")
    pass1 = 100.0 * n_passed / n_total

    out = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "model_key": model_key,
        "model_path": model_path,
        "n_total": n_total,
        "n_passed": n_passed,
        "candidate_pass_at_1_percent": pass1,
        "runtime_sec_total": time.time() - t0,
        "results": results,
    }
    run_ts = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_path = RESULTS_DIR / f"07_{model_key}_humaneval_{run_ts}.json"
    with out_path.open("w") as f:
        json.dump(out, f, indent=2)

    print(f"\nDone. {model_key} pass@1 = {pass1:.6f}")
    print(f"saved: {out_path}")
    return out

In [13]:
with BENCHMARK_PATH.open() as f:                                                                                         
    bench = json.load(f)                                                                                                 

def load_model(model):
    # first, remove existing model(s) in memory
    for name in ["model", "tokenizer", "llm", "llm_fp16", "llm_q8", "llm_q4"]:                                               
        if name in globals():                                                                                                
            del globals()[name]                                                                                              
                                                                                                                            
    gc.collect()                                                                                                             
                                                                                                                            
    try:                                                                                                                     
        import torch                                                                                                         
        if hasattr(torch, "mps") and torch.backends.mps.is_available():                                                      
            torch.mps.empty_cache()                                                                                          
    except Exception:                                                                                                        
        pass                                                                                                                 
                                                                                                                            
    print("Cleanup done.")   

    # Then, load the new model
    model_path = bench["benchmark_results"]["artifacts"][model]["artifact_path"]

    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_threads=8,
        n_gpu_layers=-1,
        verbose=False,
    )

    print(f"Loaded {model} model: {model_path}")

In [14]:
load_model("q4_k_m")

Cleanup done.


llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


Loaded q4_k_m model: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf


In [15]:
# Test humanevals on q4 gguf
run_humaneval_gguf("q4_k_m", max_problems=164)

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


[ 10/164] running pass rate: 100.00%
[ 20/164] running pass rate: 100.00%
[ 30/164] running pass rate: 93.33%
[ 40/164] running pass rate: 90.00%
[ 50/164] running pass rate: 90.00%
[ 60/164] running pass rate: 90.00%
[ 70/164] running pass rate: 87.14%
[ 80/164] running pass rate: 85.00%
[ 90/164] running pass rate: 83.33%
[100/164] running pass rate: 82.00%
[110/164] running pass rate: 80.91%
[120/164] running pass rate: 81.67%
[130/164] running pass rate: 78.46%
[140/164] running pass rate: 75.00%
[150/164] running pass rate: 74.67%
[160/164] running pass rate: 75.62%
[164/164] running pass rate: 75.00%

Done. q4_k_m pass@1 = 75.000000
saved: /Users/jarrett/dev/quantization-study/results/07_q4_k_m_humaneval_20260511T130907Z.json


{'timestamp_utc': '2026-05-11T13:09:07.463510+00:00',
 'model_key': 'q4_k_m',
 'model_path': '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf',
 'n_total': 164,
 'n_passed': 123,
 'candidate_pass_at_1_percent': 75.0,
 'runtime_sec_total': 202.24286699295044,
 'results': [{'task_id': 'HumanEval/0',
   'result': 'passed',
   'generated': '    numbers.sort()\n    for i in range(len(numbers) - 1):\n        if numbers[i + 1] - numbers[i] < threshold:\n            return True\n    return False',
   'elapsed_sec': 1.0794990062713623},
  {'task_id': 'HumanEval/1',
   'result': 'passed',
   'generated': "    result = []\n    stack = []\n    current_group = []\n\n    for char in paren_string:\n        if char == ' ':\n            continue\n        if char == '(':\n            stack.append(char)\n            current_group.append(char)\n        elif char == ')':\n            stack.pop()\n            current_group.append(char)\n            if not stack:\n  

In [16]:
load_model("q8_0")

Cleanup done.


llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


Loaded q8_0 model: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf


In [17]:
# Test humanevals on q8 gguf
run_humaneval_gguf("q8_0", max_problems=164)

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


[ 10/164] running pass rate: 100.00%
[ 20/164] running pass rate: 100.00%
[ 30/164] running pass rate: 96.67%
[ 40/164] running pass rate: 90.00%
[ 50/164] running pass rate: 88.00%
[ 60/164] running pass rate: 86.67%
[ 70/164] running pass rate: 84.29%
[ 80/164] running pass rate: 83.75%
[ 90/164] running pass rate: 82.22%
[100/164] running pass rate: 82.00%
[110/164] running pass rate: 78.18%
[120/164] running pass rate: 78.33%
[130/164] running pass rate: 75.38%
[140/164] running pass rate: 72.14%
[150/164] running pass rate: 72.00%
[160/164] running pass rate: 72.50%
[164/164] running pass rate: 71.95%

Done. q8_0 pass@1 = 71.951220
saved: /Users/jarrett/dev/quantization-study/results/07_q8_0_humaneval_20260511T131758Z.json


{'timestamp_utc': '2026-05-11T13:17:58.904263+00:00',
 'model_key': 'q8_0',
 'model_path': '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf',
 'n_total': 164,
 'n_passed': 118,
 'candidate_pass_at_1_percent': 71.95121951219512,
 'runtime_sec_total': 214.18932461738586,
 'results': [{'task_id': 'HumanEval/0',
   'result': 'passed',
   'generated': '    for i in range(len(numbers)):\n        for j in range(i + 1, len(numbers)):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n    return False',
   'elapsed_sec': 0.7529768943786621},
  {'task_id': 'HumanEval/1',
   'result': 'passed',
   'generated': "    result = []\n    stack = []\n    current_group = []\n\n    for char in paren_string:\n        if char == ' ':\n            continue\n        if char == '(':\n            stack.append(char)\n            current_group.append(char)\n        elif char == ')':\n            stack.pop()\n            current_group.app

In [18]:
load_model("fp16")

Cleanup done.


llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


Loaded fp16 model: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf


In [19]:
run_humaneval_gguf("fp16", max_problems=164)

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 0.7, mscale_all_dim = 0.7)
llama_context: n_ctx_seq (2048) < n_ctx_train (163840) -- the full capacity of the model will not be utilized


[ 10/164] running pass rate: 100.00%
[ 20/164] running pass rate: 100.00%
[ 30/164] running pass rate: 96.67%
[ 40/164] running pass rate: 90.00%
[ 50/164] running pass rate: 88.00%
[ 60/164] running pass rate: 86.67%
[ 70/164] running pass rate: 82.86%
[ 80/164] running pass rate: 83.75%
[ 90/164] running pass rate: 82.22%
[100/164] running pass rate: 83.00%
[110/164] running pass rate: 80.00%
[120/164] running pass rate: 80.00%
[130/164] running pass rate: 76.92%
[140/164] running pass rate: 72.86%
[150/164] running pass rate: 72.67%
[160/164] running pass rate: 73.12%
[164/164] running pass rate: 72.56%

Done. fp16 pass@1 = 72.560976
saved: /Users/jarrett/dev/quantization-study/results/07_fp16_humaneval_20260511T132341Z.json


{'timestamp_utc': '2026-05-11T13:23:41.123848+00:00',
 'model_key': 'fp16',
 'model_path': '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf',
 'n_total': 164,
 'n_passed': 119,
 'candidate_pass_at_1_percent': 72.5609756097561,
 'runtime_sec_total': 322.9743947982788,
 'results': [{'task_id': 'HumanEval/0',
   'result': 'passed',
   'generated': '    for i in range(len(numbers)):\n        for j in range(i + 1, len(numbers)):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n    return False',
   'elapsed_sec': 0.974153995513916},
  {'task_id': 'HumanEval/1',
   'result': 'passed',
   'generated': "    result = []\n    stack = []\n    current_group = []\n\n    for char in paren_string:\n        if char == ' ':\n            continue\n        if char == '(':\n            stack.append(char)\n            current_group.append(char)\n        elif char == ')':\n            stack.pop()\n            current_group.append(

# Sensitivity Experiments
Now that we've got some baseline results from our three models, it's time to perform sensitivity analysis.

Next steps:
1) Build a clean summary table from timestamped fp16/q8/q4 run files
2) Set the reference baseline for notebook 07
3) Do one ablation loop step (claim -> run -> record), and repeat

In [20]:
pat = re.compile(r"^07_(fp16|q8_0|q4_k_m)_humaneval_(\d{8}T\d{6}Z)\.json$")

rows = []
for p in RESULTS_DIR.glob("07_*_humaneval_*.json"):
    m = pat.match(p.name)
    if not m:
        continue
    model_key, run_ts = m.group(1), m.group(2)
    with p.open() as f:
        d = json.load(f)
    rows.append({
        "model_key": model_key,
        "run_ts": run_ts,
        "pass_at_1_percent": d.get("candidate_pass_at_1_percent"),
        "n_total": d.get("n_total"),
        "path": str(p),
    })

df = pd.DataFrame(rows).sort_values(["model_key", "run_ts"])
latest = df.groupby("model_key", as_index=False).tail(1).sort_values("pass_at_1_percent", ascending=False)

print("Latest run per model:")
display(latest[["model_key", "run_ts", "pass_at_1_percent", "n_total"]])

latest_path_map = dict(zip(latest["model_key"], latest["path"]))
print("\nLatest file map:")
print(latest_path_map)

Latest run per model:


,model_key,run_ts,pass_at_1_percent,n_total
1,q4_k_m,20260511T130907Z,75.000000,164
2,fp16,20260511T132341Z,72.560976,164
0,q8_0,20260511T131758Z,71.951220,164



Latest file map:
{'q4_k_m': '/Users/jarrett/dev/quantization-study/results/07_q4_k_m_humaneval_20260511T130907Z.json', 'fp16': '/Users/jarrett/dev/quantization-study/results/07_fp16_humaneval_20260511T132341Z.json', 'q8_0': '/Users/jarrett/dev/quantization-study/results/07_q8_0_humaneval_20260511T131758Z.json'}


In [21]:
# Step 2: save a notebook-07 parity baseline summary (llama.cpp-only reference).
baseline_07 = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "harness": {
        "backend": "llama_cpp",
        "dataset": "openai/openai_humaneval",
        "n_total": 164,
        "notes": "Parity runs under same backend/settings; use fp16 here as Notebook-07 reference, not notebook-03 transformers bsaeline.",
    },
    "latest_runs": {
        "q4_k_m": {
            "run_ts": "20260407T063833Z",
            "pass_at_1_percent": 75.0,
            "path": str(RESULTS_DIR / "07_q4_k_m_humaneval_20260407T063833Z.json")
        },
        "fp16": {
            "run_ts": "20260407T064700Z",
            "pass_at_1_percent": 72.560976,
            "path": str(RESULTS_DIR / "07_fp16_humaneval_20260407T064700Z.json"),
        },
        "q8_0": {
            "run_ts": "20260407T064158Z",
            "pass_at_1_percent": 71.951220,
        },
    },
    "ranking": ["q4_k_m", "fp16", "q8_0"],
}

out_path = RESULTS_DIR / "07_parity_baseline_summary.json"
with out_path.open("w") as f:
    json.dump(baseline_07, f, indent=2)

print(f"Saved: {out_path}")
print("Reference fp16 pass@1 for Notebook-07 sensitivity deltas:", baseline_07["latest_runs"]["fp16"]["pass_at_1_percent"])

Saved: /Users/jarrett/dev/quantization-study/results/07_parity_baseline_summary.json
Reference fp16 pass@1 for Notebook-07 sensitivity deltas: 72.560976


In [22]:
# Step 3: update sensitivity manifest reference baseline to notebook-07 llama.cpp fp16 parity baseline.
PARITY_PATH = RESULTS_DIR / "07_parity_baseline_summary.json"

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

with PARITY_PATH.open() as f:
    parity = json.load(f)

new_ref = float(parity["latest_runs"]["fp16"]["pass_at_1_percent"])
old_ref = float(manifest["reference"]["baseline_pass_at_1_percent"])

manifest["reference"]["baseline_pass_at_1_percent"] = new_ref
manifest["reference"]["baseline_source"] = str(PARITY_PATH)
manifest["reference"]["baseline_kind"] = "fp16_llama_cpp_parity"
manifest["reference"]["updated_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()

for section in ["layer", "expert"]:
    for a in manifest["ablations"][section]:
        cand = a.get("candidate_pass_at_1_percent")
        if cand is not None:
            delta = new_ref - float(cand)
            a["delta_pass_at_1_percent"] = delta
            a["relative_drop"] = (delta / new_ref) if new_ref else None

with MANIFEST_PATH.open("w") as f:
    json.dump(manifest, f, indent=2)

print(f"Updated manifest baseline: {old_ref:.6f} -> {new_ref:.6f}")

Updated manifest baseline: 60.365854 -> 72.560976


In [23]:
# Step 4: claim next pending ablation
# Mark stale in_progress as completed if it already has a recorded candidate score, then claim next pending.
with MANIFEST_PATH.open() as f:
    m = json.load(f)

all_items = m["ablations"]["layer"] + m["ablations"]["expert"]

# close stale run if score already exists
for a in all_items:
    if a["status"] == "in_progress" and a.get("candidate_pass_at_1_percent") is not None:
        a["status"] = "completed"
        a["completed_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()

# claim next pending
active = [a for a in all_items if a["status"] == "in_progress"]
if active:
    print("Still in progress:", active[0]["ablation_id"])
else:
    pending = [a for a in all_items if a["status"] == "pending"]
    if not pending:
        print("No pending ablations left.")
    else:
        nxt = pending[0]
        nxt["status"] = "in_progress"
        nxt["started_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
        print("Claimed:", nxt["ablation_id"], "|", nxt["kind"])

with MANIFEST_PATH.open("w") as f:
    json.dump(m, f,)

Claimed: layer_group_01 | layer_group


In [24]:
# Cancel stale active ablation and claim next pending.                                                                                                                                                                                         
with MANIFEST_PATH.open() as f:                                                                                        
    m = json.load(f)                                                                                                   
                                                                                                                        
all_items = m["ablations"]["layer"] + m["ablations"]["expert"]                                                         
                                                                                                                        
# Cancel any currently in-progress item (stale pre-parity run).                                                        
for a in all_items:                                                                                                    
    if a["status"] == "in_progress":                                                                                   
        a["status"] = "cancelled"                                                                                      
        a["cancelled_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()                                           
        a["notes"] = (a.get("notes", "") + " cancelled after harness/baseline reset").strip()                          
                                                                                                                        
# Claim next pending.                                                                                                  
pending = [a for a in all_items if a["status"] == "pending"]                                                           
if not pending:                                                                                                        
    print("No pending ablations left.")                                                                                
else:                                                                                                                  
    nxt = pending[0]                                                                                                   
    nxt["status"] = "in_progress"                                                                                      
    nxt["started_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()                                               
    print("Claimed:", nxt["ablation_id"], "|", nxt["kind"], "| layers:", nxt["target_layers"], "| experts:",           
nxt["target_experts"])                                                                                                 
                                                                                                                        
with MANIFEST_PATH.open("w") as f:                                                                                     
    json.dump(m, f, indent=2)  

Claimed: layer_group_02 | layer_group | layers: [6, 7, 8] | experts: None
